In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import matplotlib.pyplot as plt

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve
from src.curves.zero_curve import ZeroCurve

from src.instruments.instrument_builder import InstrumentBuilder

from src.trades.interest_rate_swap import InterestRateSwap

from src.pricing.swap_pricer import SwapPricer

from src.curves.simulator.hull_white import HullWhiteSimulator

from src.risk.exposure.stochastic_exposure_engine import MonteCarloExposureEngine

In [2]:
# downloading market curves
market_loader = MarketLoader()
market_curves = market_loader.market_loader_pipeline()

# downloading swap curves
swap_loader = MarketLoader()
swap_curves = swap_loader.swap_loader_pipeline()

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..
usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


In [3]:
### create curve snapshots
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

In [4]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)

### discount, projection and zero curve builder
# bootstrapping engine for generating the discount curve
all_instruments = deposit_instruments + future_instruments + ois_instruments

engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

# projection curve
projection_curve = ProjectionCurve(discount_curve = discount_curve)

# zero curve
zero_curve = ZeroCurve(discount_curve = discount_curve)

In [5]:
# sample IRS trade objects
IR_swap = InterestRateSwap(
    notional = 1_000_000,
    maturity = 3.0,
    fixed_rate = 3.40,
    pay_fixed = True
)

In [6]:
### exposure risk analytics
# pricer
pricer = SwapPricer(
    discount_curve = discount_curve,
    projection_curve = projection_curve
)

pv = pricer.price(swap = IR_swap)
print(f'PV = {pv:.2f}')

# simulator
simulator = HullWhiteSimulator(
    r0 = min(zero_curve.zero_rates.values()),
    random_seed = 12
)

# exposure engine
mc_engine = MonteCarloExposureEngine(
    pricer = pricer,
    simulator = simulator,
    zero_curve = zero_curve
)

PV = 7858.41


In [7]:
# expected exposure
ee = mc_engine.expected_exposure(
    swap = IR_swap, 
    n_paths = 250
)
ee

,Times,EE
0,0.00,7858.414276
1,0.25,5213.738044
2,0.50,11271.758664
3,0.75,6368.467495
4,1.00,11143.132287
5,1.25,6171.274082
6,1.50,9563.576422
7,1.75,3846.301846
8,2.00,6631.007614
9,2.25,1336.118116


In [8]:
# expected negative exposure
ene = mc_engine.expected_negative_exposure(
    swap = IR_swap,
    n_paths = 250
)
ene

,Times,ENE
0,0.00,-0.000000
1,0.25,5160.683156
2,0.50,3511.370397
3,0.75,8034.765974
4,1.00,4198.487240
5,1.25,8332.450265
6,1.50,4329.064670
7,1.75,7742.593890
8,2.00,3369.097267
9,2.25,7726.475982


In [9]:
# potential future exposure
perc = 95.0

pfe = mc_engine.potential_future_exposure(
    swap = IR_swap,
    percentile = perc,
    n_paths = 250
)
pfe

,Times,PFE_95%
0,0.00,7858.414276
1,0.25,22380.321681
2,0.50,36307.272149
3,0.75,28350.107497
4,1.00,35627.378969
5,1.25,25074.591115
6,1.50,29840.082000
7,1.75,18337.044488
8,2.00,23863.939928
9,2.25,9680.491793


In [10]:
# expected positive exposure
epe = mc_engine.expected_positive_exposure(
    swap = IR_swap,
    n_paths = 250
)
epe

5675.052557846324

In [11]:
mc_engine.mc_exposure_report(swap = IR_swap, n_paths = 250)

,Times,EE,ENE,PFE_95%
0,0.00,7858.414276,-0.000000,7858.414276
1,0.25,4544.798815,6345.299693,19215.842982
2,0.50,10517.946033,3585.781435,35199.015079
3,0.75,6467.529835,7754.146962,26104.206669
4,1.00,11067.433567,4032.176654,33739.849506
5,1.25,5647.827452,8309.715943,29382.814103
6,1.50,9633.697545,4469.520001,34673.614861
7,1.75,4236.097804,7819.644099,20944.004074
8,2.00,7031.933292,3474.567858,25833.662557
9,2.25,1650.985520,7684.062991,11718.865256
